## _PyG to NetworkX_

In [1]:
import sys, os, glob, yaml

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
import pprint
from tqdm import tqdm
import trackml.dataset

In [4]:
import torch
from torch_geometric.data import Data

In [5]:
pp = pprint.PrettyPrinter(indent=2)

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['EXATRKX_DATA'] = os.path.abspath(os.curdir)

### _PyG Data from GNN Stage_

In [7]:
# Read Event from the Testset
inputdir="run/gnn_evaluation/test"
outputdir="run/stt2_trkx/trkx_from_cc"
os.makedirs(outputdir, exist_ok=True)

In [8]:
all_files = glob.glob(os.path.join(inputdir, "*"))
all_files = sorted(all_files)
print("Total Test Events: ", len(all_files))

Total Test Events:  5000


In [9]:
all_files[:10]

['run/gnn_evaluation/test/5000',
 'run/gnn_evaluation/test/5001',
 'run/gnn_evaluation/test/5002',
 'run/gnn_evaluation/test/5003',
 'run/gnn_evaluation/test/5004',
 'run/gnn_evaluation/test/5005',
 'run/gnn_evaluation/test/5006',
 'run/gnn_evaluation/test/5007',
 'run/gnn_evaluation/test/5008',
 'run/gnn_evaluation/test/5009']

In [10]:
# Use One Event
filename = all_files[1]
evtid = int(os.path.basename(filename))
print("evtid: ", evtid)

evtid:  5001


In [11]:
data = torch.load(filename, map_location=device)

### _Explore the `PyD::Data`_

In [12]:
print(data.keys)

['hid', 'y_pid', 'pid', 'layers', 'modulewise_true_edges', 'event_file', 'edge_index', 'pt', 'layerwise_true_edges', 'x', 'scores']


In [13]:
data.num_nodes, data.num_edges, data.num_features

(171, 2, 3)

In [14]:
data.is_directed()

True

In [15]:
'edge_attr' in data

False

In [16]:
# for key, item in data:
#    print(f'{key}, {item} found in Data')

In [17]:
data

Data(x=[171, 3], pid=[171], layers=[171], event_file='/global/cscratch1/sd/aakram/train_all/event0000095001', hid=[171], pt=[171], modulewise_true_edges=[2, 160], layerwise_true_edges=[2, 172], edge_index=[2, 808], y_pid=[808], scores=[1616])

In [18]:
# data.x or data['x']

In [19]:
# see https://stackoverflow.com/questions/19984102/select-elements-of-numpy-array-via-boolean-mask-array
true_edges = data.edge_index[:,data.y_pid]
true_edges.shape

torch.Size([2, 172])

In [20]:
false_edges = data.edge_index[:,~data.y_pid]
false_edges.shape

torch.Size([2, 636])

### _(a) PyG to NetworkX_

Using `to_networkx()` from `torch_geometric.utils`.

In [21]:
import networkx as nx
from torch_geometric.utils import to_networkx

In [22]:
# from https://github.com/exatrkx/exatrkx-neurips19/blob/master/gnn-tracking/heptrkx/nx_graph/utils_plot.py
def get_pos(Gp):
    pos = {}
    for node in Gp.nodes():
        # r, phi, z = Gp.node[node]['pos'][:3]
        r,phi = G.nodes[node]['x'][:2]
        x = r * np.cos(phi)
        y = r * np.sin(phi)
        pos[node] = np.array([x, y])
    return pos

def plot_networkx(G, ax=None, only_true=False):
    """G is networkx graph,
    node feature: {'pos': [r, phi, z]}
    edge feature: {"solution": []}
    """
    if ax is None:
        fig, ax = plt.subplots()

    n_edges = len(G.edges())
    edge_colors = [0.]*n_edges
    true_edges = []
    for iedge,edge in enumerate(G.edges(data=True)):
        if int(edge[2]['y_pid']) == 1:
            edge_colors[iedge] = 'r'
            true_edges.append((edge[0], edge[1]))
        else:
            edge_colors[iedge] = 'grey'

    Gp = nx.edge_subgraph(G, true_edges) if only_true else G
    edge_colors = ['k']*len(true_edges) if only_true else edge_colors 

    pos = get_pos(Gp)

    nx.draw(Gp, pos, node_color='#A0CBE2', edge_color=edge_colors,
       width=0.5, with_labels=False, node_size=1, ax=ax, arrows=False)

In [46]:
# Data is a DiGraph i.e. G := DiGraph()
# G = to_networkx(data)
G = to_networkx(data, node_attrs=['x'], edge_attrs=['scores', 'y_pid'])

In [53]:
# Shows nodes with attributes e.g. attrs = x [x,y,ir]
# G.nodes(data=True)
# G.nodes.data()[0]

In [60]:
# Shows edges with attributes e.g (edges, {attrs}) >> (0, 10, {'scores': .., 'y_pid': ..})
# G.edges(data=True)
# G.edges.data()
# G.edges.data("scores")

In [55]:
G.in_edges == G.out_edges

True

In [63]:
G.edges[(0,10)]["scores"]

0.9999220371246338

In [26]:
# plot_networkx(G)

- Components Methods for DiGraphs
- DiGraph has `weakly` or `strongly` connected components

In [27]:
# Strongly Connected Components (SCCs)
nx.is_strongly_connected(G)

False

In [28]:
# Weakly Connected Components (WCCs)
nx.is_weakly_connected(G)

True

In [29]:
# WCC from DiGraph
# clusters = list(map(lambda g: g, nx.weakly_connected_components(G)))
# clusters = [g for g in nx.weakly_connected_components(G)]

In [30]:
# OR use loop to build list
clusters = []
for g in nx.weakly_connected_components(G):
    clusters.append(g)
    print(g, "\n")

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170} 



### _(b) - PyG to NetworkX_

Using `add_weighted_edges_from()` from the NetworkX for a `DiGraph()`

In [31]:
# DiGraph
G = nx.DiGraph()

In [32]:
scores = data.scores[:data.edge_index.shape[1]].cpu().detach().numpy()  # score has twice the size of edge_index (flip(0) was used)
senders = data.edge_index[0].cpu().detach().numpy()
receivers = data.edge_index[1].cpu().detach().numpy()

In [33]:
senders.shape, receivers.shape, scores.shape

((808,), (808,), (808,))

In [34]:
type(senders), type(receivers), type(scores)

(numpy.ndarray, numpy.ndarray, numpy.ndarray)

In [35]:
senders[:5]

array([0, 0, 0, 1, 1])

In [36]:
receivers[:5]

array([10, 15, 17, 12, 14])

In [37]:
scores[:5]

array([9.9992204e-01, 8.0207006e-05, 2.0691472e-04, 9.9992442e-01,
       8.0039368e-05], dtype=float32)

In [38]:
# Create list of tuples '(u, v, w)' from senders, receivers and scores ndarrays
elist = list(map(lambda u, v, w:(u, v, w), senders, receivers, scores))

In [39]:
elist[:5]

[(0, 10, 0.99992204),
 (0, 15, 8.0207006e-05),
 (0, 17, 0.00020691472),
 (1, 12, 0.9999244),
 (1, 14, 8.003937e-05)]

In [40]:
G.edges

OutEdgeView([])

In [41]:
G.edges.items()

ItemsView(OutEdgeView([]))

In [42]:
G.edges.values()

ValuesView(OutEdgeView([]))

In [43]:
G.edges.data()

OutEdgeDataView([])

In [44]:
# Doesn't work
# nx.is_weakly_connected(G)

NetworkXPointlessConcept: Connectivity is undefined for the null graph.

In [ ]:
clusters = []
for g in nx.weakly_connected_components(G):
    clusters.append(g)
    print(g, "\n")

In [ ]:
# Draw Graph
# nx.draw(G, with_labels=True)
nx.draw(G)
plt.show()